# Regelungstechnik Schnellworkflow (Notebook)

Dieses Notebook ist auf schnelle Anwendung im Vorlesungsablauf ausgelegt: Modell -> Stabilitaet -> Entwurf -> Verifikation.

In [ ]:
import numpy as np
from regelungstechnik import (
    laplace_transform, inverse_laplace, partialbruchzerlegung,
    reihenschaltung, parallelschaltung, rueckkopplung,
    hurwitz_kriterium, routh_kriterium, nyquist_kriterium,
    reglerparameter_nach_verfahren, phasenkorrekturglied_auslegung,
    wurzelortsauslegung,
    plot_bode, plot_nyquist, plot_wurzelortskurve, plot_sprungantwort
)

## 1) Grundfunktionen im s-Bereich (Laplace / Partialbruch)

In [ ]:
import sympy as sp
t, s = sp.symbols('t s')
f_t = sp.exp(-2*t)
print('Laplace:', laplace_transform(f_t)['ergebnis'])
print('Inverse Laplace:', inverse_laplace(1/(s+3))['ergebnis'])
print('Partialbruch:', partialbruchzerlegung([1, 2], [1, 3, 2])['ergebnis'])

## 2) Blockschaltbildrechnung und geschlossener Kreis

In [ ]:
# Beispielstrecke G(s) = 1/(s^2 + 3s + 2)
G_num, G_den = [1.0], [1.0, 3.0, 2.0]
# Einheitliche Rueckfuehrung
T_num, T_den = rueckkopplung((G_num, G_den), ([1.0], [1.0]), negativ=True)['ergebnis']
print('Geschlossener Kreis num:', T_num)
print('Geschlossener Kreis den:', T_den)

## 3) Stabilitaet (Hurwitz, Routh, allgemeines Nyquist)

In [ ]:
print('Hurwitz stabil:', hurwitz_kriterium(T_den)['ergebnis']['stabil'])
print('Routh stabil:', routh_kriterium(T_den)['ergebnis']['stabil'])
nyq = nyquist_kriterium(G_num, G_den, w_min=1e-3, w_max=1e3, punkte=3000)
print('Nyquist stabil:', nyq['ergebnis']['stabil'])
print('Nyquist Kennzahlen: P={P}, N_cw={N_cw}, Z={Z}'.format(**nyq['ergebnis']))
plot_nyquist(G_num, G_den, w_min=1e-3, w_max=1e3, punkte=3000)['plot_pfad']

## 4) Entwurf (Wurzelort, ZN/CC, Phasenkorrekturglied)

In [ ]:
auslegung = wurzelortsauslegung(G_num, G_den, k_start=0.0, k_ende=20.0, anzahl_k=81)
print('Empfohlene k-Werte:', auslegung['ergebnis']['k_empfohlen'])
plot_wurzelortskurve(G_num, G_den, k_bereich=auslegung['ergebnis']['k_werte'],
                   k_markierungen=auslegung['ergebnis']['k_empfohlen'])['plot_pfad']

zn = reglerparameter_nach_verfahren('PID', 'ziegler-nichols', modus='offen', K=1.0, T=1.0, K_T=0.2)
print('ZN PID:', zn['ergebnis'])

lead = phasenkorrekturglied_auslegung('anhebend', phi_grad=35.0, omega_c=2.0, K=1.0)
lag = phasenkorrekturglied_auslegung('absenkend', phi_grad=20.0, omega_c=2.0, K=1.0)
print('Lead num/den:', lead['ergebnis']['num'], lead['ergebnis']['den'])
print('Lag  num/den:', lag['ergebnis']['num'], lag['ergebnis']['den'])

## 5) Verifikation (Bode, Sprungantwort)
Nutze die erzeugten Regler-/Korrekturgliedparameter und pruefe danach Frequenz- und Zeitbereich.

In [ ]:
plot_bode(G_num, G_den)['plot_pfad'], plot_sprungantwort(G_num, G_den, t_ende=10.0)['plot_pfad']